# 1. Tiêu đề / giới thiệu

Dự đoán giá nhà bằng KNN Regression

Notebook này hoàn thành Bước 2: xây dựng Machine Learning KNN Regression trên Google Colab cho bài toán dự đoán giá nhà. Phạm vi chỉ gồm dataset, train/test split, StandardScaler, KNeighborsRegressor, thử nhiều giá trị K, đánh giá, save/load model và predict thử.

## 2. Cài đặt thư viện

Cell này kiểm tra và cài các thư viện Machine Learning cần thiết nếu môi trường Colab chưa có sẵn. Không cài FastAPI, Uvicorn hoặc pyngrok trong Bước 2.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Tất cả thư viện Machine Learning cần thiết đã được cài sẵn.")

## 3. Import thư viện

Import các thư viện cần dùng cho xử lý dữ liệu, train model, đánh giá metrics và lưu model.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 4. Tạo dataset

Tạo synthetic dataset gồm area, rooms, distance và price. Price được tạo theo quan hệ gần tuyến tính có Gaussian noise để dữ liệu không hoàn hảo tuyệt đối.

In [ ]:
random_seed = 42
n_samples = 1000
rng = np.random.default_rng(random_seed)

area = rng.uniform(30, 250, n_samples)
rooms = rng.integers(1, 7, n_samples)
distance = rng.uniform(0.5, 30, n_samples)
noise = rng.normal(0, 150, n_samples)

price = 500 + 18 * area + 220 * rooms - 30 * distance + noise
price = np.maximum(price, 300)

df = pd.DataFrame(
    {
        "area": np.round(area, 2),
        "rooms": rooms,
        "distance": np.round(distance, 2),
        "price": np.round(price, 2),
    }
)

print(f"Đã tạo dataset gồm {len(df)} mẫu dữ liệu.")

## 5. Khám phá dữ liệu

Kiểm tra nhanh 5 dòng đầu, kích thước dữ liệu, thông tin cột, thống kê mô tả, missing values và kiểu dữ liệu.

In [ ]:
display_columns_vi = {
    "area": "Diện tích (m2)",
    "rooms": "Số phòng",
    "distance": "Khoảng cách tới trung tâm (km)",
    "price": "Giá nhà (triệu VND)",
}

print("5 dòng đầu tiên của dataset:")
display(df.head().rename(columns=display_columns_vi))

print(f"Kích thước dataset: {df.shape[0]} dòng, {df.shape[1]} cột")

print("\nThông tin DataFrame:")
df.info()

print("\nThống kê mô tả:")
display(df.describe().rename(columns=display_columns_vi))

data_quality = pd.DataFrame(
    {
        "Số giá trị thiếu": df.isna().sum(),
        "Kiểu dữ liệu": df.dtypes.astype(str),
    }
).rename(index=display_columns_vi)
print("\nKiểm tra dữ liệu thiếu và kiểu dữ liệu:")
display(data_quality)

## 6. Tạo X / y

`X` là dữ liệu đầu vào gồm area, rooms và distance. `y` là giá nhà cần dự đoán.

In [ ]:
feature_columns = ["area", "rooms", "distance"]
target_column = "price"

X = df[feature_columns]
y = df[target_column]

print("Các feature đầu vào:", ", ".join(display_columns_vi[column] for column in feature_columns))
print("Target cần dự đoán:", display_columns_vi[target_column])
print(f"Kích thước X: {X.shape}")
print(f"Kích thước y: {y.shape}")

## 7. Chia train/test

Chia dữ liệu thành 80% train và 20% test với random_state = 42 để kết quả có thể tái lập.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_seed,
)

print(f"Số mẫu train: {len(X_train)}")
print(f"Số mẫu test: {len(X_test)}")

## 8. Thử nhiều giá trị K

Thử nhiều giá trị K. Mỗi model là một Pipeline gồm StandardScaler và KNeighborsRegressor, vì KNN dựa trên khoảng cách nên cần scale feature.

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15]
metric_rows = []

for k in k_values:
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("knn", KNeighborsRegressor(n_neighbors=k)),
        ]
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    metric_rows.append(
        {
            "k": k,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
        }
    )

results_df = pd.DataFrame(metric_rows)
print("Đã train và đánh giá xong tất cả giá trị K.")

## 9. Compare metrics

Bang duoi day hien thi MAE, RMSE va R2 cua tung gia tri K. RMSE cang thap thi sai so trung binh cang tot.

In [ ]:
display(results_df.sort_values("k").round(4))

ranked_results_df = results_df.sort_values(["rmse", "mae", "k"]).reset_index(drop=True)
display(ranked_results_df.round(4))

## 10. Select best K

Chon K tot nhat dua tren RMSE thap nhat. Neu co nhieu K gan tuong duong, uu tien K khong qua nho de model on dinh hon.

In [ ]:
min_rmse = results_df["rmse"].min()
near_best_threshold = min_rmse * 1.01
near_best_df = results_df[results_df["rmse"] <= near_best_threshold].copy()

stable_candidates_df = near_best_df[near_best_df["k"] >= 5]
if stable_candidates_df.empty:
    selected_row = ranked_results_df.iloc[0]
else:
    selected_row = stable_candidates_df.sort_values(["rmse", "mae", "k"]).iloc[0]

best_k = int(selected_row["k"])
best_mae = float(selected_row["mae"])
best_rmse = float(selected_row["rmse"])
best_r2 = float(selected_row["r2"])

display(results_df.sort_values("k").round(4))
print(f"Best K: {best_k}")
print(f"Best MAE: {best_mae:.2f} million VND")
print(f"Best RMSE: {best_rmse:.2f} million VND")
print(f"Best R2: {best_r2:.4f}")
print(
    "Selected K explanation: K was selected from the lowest RMSE group; "
    "when scores were within 1%, K >= 5 was preferred to avoid an overly sensitive neighbor setting."
)

## 11. Train final Pipeline

Train final Pipeline voi best_k tren tap train. Pipeline gom StandardScaler va KNeighborsRegressor.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(n_neighbors=best_k)),
    ]
)

final_model.fit(X_train, y_train)
print(f"Final model trained with K = {best_k}.")

## 12. Evaluate final model

Danh gia final model tren tap test bang MAE, RMSE va R2.

In [ ]:
final_predictions = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, final_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
final_r2 = r2_score(y_test, final_predictions)

print("Final model metrics")
print(f"MAE : {final_mae:.2f} million VND")
print(f"RMSE: {final_rmse:.2f} million VND")
print(f"R2  : {final_r2:.4f}")

model_summary = pd.DataFrame(
    [
        {"metric": "MAE", "value": final_mae},
        {"metric": "RMSE", "value": final_rmse},
        {"metric": "R2", "value": final_r2},
    ]
)
display(model_summary.round(4))

KNN model characteristics

KNN la instance-based learning nen khong hoc coefficient/intercept nhu Linear Regression. Thay vao do, model du doan dua tren cac diem lang gieng gan nhat trong khong gian feature da scale.

In [ ]:
knn_characteristics = {
    "best_k": best_k,
    "train_samples": len(X_train),
    "feature_count": X_train.shape[1],
    "final_mae": final_mae,
    "final_rmse": final_rmse,
    "final_r2": final_r2,
}

display(pd.DataFrame([knn_characteristics]).round(4))

## 13. Save model

Luu toan bo Pipeline bang joblib vao working directory cua Colab voi ten `knn_house_model.pkl`, sau do kiem tra file ton tai.

In [ ]:
model_path = Path("knn_house_model.pkl")
joblib.dump(final_model, model_path)

if model_path.exists():
    print(f"Model saved successfully: {model_path.resolve()}")
else:
    raise FileNotFoundError(f"Model file was not created: {model_path}")

## 14. Load model

Load lai model bang joblib de dam bao file model co the dung cho buoc inference.

In [ ]:
loaded_model = joblib.load(model_path)
print("Model loaded successfully.")
print(type(loaded_model))

## 15. Test prediction

Tao input mau voi dung thu tu feature: area, rooms, distance. Dung loaded model de predict gia nha.

In [ ]:
sample_input = {
    "area": 85.5,
    "rooms": 3,
    "distance": 5.2,
}

sample_df = pd.DataFrame([sample_input], columns=feature_columns)
predicted_price = float(loaded_model.predict(sample_df)[0])

prediction_result = {
    "area": sample_input["area"],
    "rooms": sample_input["rooms"],
    "distance": sample_input["distance"],
    "predicted_price": round(predicted_price, 2),
    "unit": "million VND",
}

display(pd.DataFrame([prediction_result]))
print(
    f"Predicted price for area={sample_input['area']} m2, "
    f"rooms={sample_input['rooms']}, distance={sample_input['distance']} km: "
    f"{predicted_price:.2f} million VND"
)

## 16. Conclusion

- Bai toan trong notebook nay la KNN Regression cho du doan gia nha.
- StandardScaler can thiet vi KNN dung khoang cach giua cac diem du lieu.
- Notebook da thu nhieu gia tri K: 1, 3, 5, 7, 9, 11 va 15.
- Notebook da chon `best_k` dua tren RMSE thap nhat, co uu tien K on dinh khi ket qua gan tuong duong.
- Final model da duoc danh gia bang MAE, RMSE va R2.
- Pipeline da duoc save/load thanh cong bang joblib.
- Model da predict thu voi input mau va san sang cho Buoc 3: FastAPI Model Server + Ngrok.

## 17. Cài thư viện cho máy chủ mô hình

Cài thêm các thư viện cần thiết cho Bước 3: FastAPI, Uvicorn và pyngrok. Không thêm backend local hoặc Docker ở bước này.

In [ ]:
server_packages = {
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "pyngrok": "pyngrok",
}

missing_server_packages = [
    package_name
    for module_name, package_name in server_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_server_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_server_packages])
    print("Đã cài xong các thư viện cho máy chủ mô hình.")
else:
    print("Các thư viện cho máy chủ mô hình đã được cài sẵn.")

## 18. Import thư viện cho máy chủ mô hình

Import các thư viện để tạo API, validate request, chạy Uvicorn trong background thread và public API bằng Ngrok.

In [ ]:
import json
import threading
import time
import urllib.error
import urllib.request
from getpass import getpass

import uvicorn
from fastapi import FastAPI
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError

## 19. Tạo schema request và response

API public dùng tên trường tiếng Việt không dấu. Model vẫn giữ feature nội bộ là `area`, `rooms`, `distance`.

In [ ]:
class HousePredictionRequest(BaseModel):
    dien_tich: float = Field(
        ...,
        gt=0,
        description="Diện tích căn nhà, đơn vị m²",
        examples=[85.5],
    )
    so_phong: int = Field(
        ...,
        ge=1,
        description="Số phòng của căn nhà",
        examples=[3],
    )
    khoang_cach_trung_tam: float = Field(
        ...,
        ge=0,
        description="Khoảng cách từ căn nhà tới trung tâm, đơn vị km",
        examples=[5.2],
    )


class HousePredictionResponse(BaseModel):
    gia_du_doan: float = Field(..., description="Giá nhà dự đoán")
    don_vi: str = Field("trieu_vnd", description="Đơn vị của giá dự đoán")


class SamplePredictionResponse(BaseModel):
    du_lieu_dau_vao: HousePredictionRequest = Field(..., description="Dữ liệu đầu vào mẫu")
    gia_du_doan: float = Field(..., description="Giá nhà dự đoán")
    don_vi: str = Field("trieu_vnd", description="Đơn vị của giá dự đoán")

## 20. Tạo FastAPI Model Server

Tạo ứng dụng FastAPI với Swagger tiếng Việt. Endpoint `/predict` chỉ dùng model đã load và không train lại model.

In [ ]:
app = FastAPI(
    title="KNN Dự đoán giá nhà",
    description="API sử dụng mô hình K-Nearest Neighbors Regression để dự đoán giá nhà.",
    version="1.0.0",
)


@app.exception_handler(RequestValidationError)
async def xu_ly_loi_du_lieu(request, exc):
    chi_tiet = []
    for loi in exc.errors():
        truong = ".".join(str(phan) for phan in loi.get("loc", []) if phan != "body")
        chi_tiet.append(
            {
                "truong": truong,
                "thong_bao": "Giá trị không hợp lệ. Điều kiện: dien_tich > 0, so_phong >= 1, khoang_cach_trung_tam >= 0.",
            }
        )
    return JSONResponse(
        status_code=422,
        content={"loi": "Dữ liệu đầu vào không hợp lệ.", "chi_tiet": chi_tiet},
    )


def du_doan_bang_model(dien_tich: float, so_phong: int, khoang_cach_trung_tam: float) -> float:
    input_data = pd.DataFrame(
        [
            {
                "area": dien_tich,
                "rooms": so_phong,
                "distance": khoang_cach_trung_tam,
            }
        ],
        columns=["area", "rooms", "distance"],
    )
    return float(loaded_model.predict(input_data)[0])


@app.get(
    "/",
    summary="Trang thông tin máy chủ mô hình",
    description="Hiển thị thông tin tổng quan và các đường dẫn API của máy chủ mô hình KNN.",
)
def trang_chu():
    return {
        "trang_thai": "hoat_dong",
        "ten_mo_hinh": "KNN du doan gia nha",
        "mo_ta": "Mo hinh K-Nearest Neighbors Regression du doan gia nha",
        "cac_duong_dan": {
            "kiem_tra_may_chu": "/health",
            "du_doan_mau": "/predict",
            "tai_lieu_api": "/docs",
        },
    }


@app.get(
    "/health",
    summary="Kiểm tra trạng thái máy chủ",
    description="Trả về trạng thái hoạt động của máy chủ mô hình KNN.",
)
def kiem_tra_trang_thai():
    return {
        "trang_thai": "hoat_dong",
        "dich_vu": "may_chu_mo_hinh_knn_du_doan_gia_nha",
    }


@app.get(
    "/predict",
    response_model=SamplePredictionResponse,
    summary="Dự đoán giá nhà với dữ liệu mẫu",
    description="Dùng dữ liệu mẫu mặc định để dự đoán giá nhà khi mở trực tiếp endpoint bằng trình duyệt.",
)
def du_doan_mau():
    du_lieu_dau_vao = HousePredictionRequest(
        dien_tich=85.5,
        so_phong=3,
        khoang_cach_trung_tam=5.2,
    )
    gia_du_doan = du_doan_bang_model(
        du_lieu_dau_vao.dien_tich,
        du_lieu_dau_vao.so_phong,
        du_lieu_dau_vao.khoang_cach_trung_tam,
    )
    return {
        "du_lieu_dau_vao": {
            "dien_tich": du_lieu_dau_vao.dien_tich,
            "so_phong": du_lieu_dau_vao.so_phong,
            "khoang_cach_trung_tam": du_lieu_dau_vao.khoang_cach_trung_tam,
        },
        "gia_du_doan": round(gia_du_doan, 2),
        "don_vi": "trieu_vnd",
    }


@app.post(
    "/predict",
    response_model=HousePredictionResponse,
    summary="Dự đoán giá nhà",
    description="Nhận diện tích, số phòng và khoảng cách tới trung tâm để dự đoán giá nhà bằng KNN Regression.",
)
def du_doan_gia_nha(request: HousePredictionRequest):
    gia_du_doan = du_doan_bang_model(
        request.dien_tich,
        request.so_phong,
        request.khoang_cach_trung_tam,
    )
    return {"gia_du_doan": round(gia_du_doan, 2), "don_vi": "trieu_vnd"}


print("Đã tạo FastAPI Model Server với endpoint GET /, GET /health, GET /predict và POST /predict.")

## 21. Chạy máy chủ bằng Uvicorn

Chạy FastAPI trên port 8000 bằng background thread để notebook vẫn tiếp tục chạy được. Nếu chạy lại cell, server cũ sẽ được yêu cầu dừng trước.

In [ ]:
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

uvicorn_config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
uvicorn_server = uvicorn.Server(uvicorn_config)


def chay_may_chu_mo_hinh():
    uvicorn_server.run()


server_thread = threading.Thread(target=chay_may_chu_mo_hinh, daemon=True)
server_thread.start()
time.sleep(2)

print("Máy chủ mô hình đang chạy trên port 8000.")
print("Địa chỉ nội bộ trong Colab: http://127.0.0.1:8000")
print("Swagger nội bộ: http://127.0.0.1:8000/docs")

## 22. Cấu hình Ngrok

Nhập Ngrok auth token bằng `getpass` để không hard-code token. Cell này đóng tunnel cũ nếu có, sau đó public port 8000.

In [ ]:
NGROK_AUTHTOKEN = getpass("Nhập NGROK_AUTHTOKEN: ").strip()

if not NGROK_AUTHTOKEN:
    raise ValueError("NGROK_AUTHTOKEN không được để trống.")

ngrok.set_auth_token(NGROK_AUTHTOKEN)
ngrok.kill()

try:
    public_tunnel = ngrok.connect(8000, "http")
    public_url = public_tunnel.public_url
except PyngrokNgrokError:
    print("Không public được máy chủ bằng Ngrok.")
    print("Nguyên nhân thường gặp: NGROK_AUTHTOKEN sai, bị thiếu, có dấu cách thừa hoặc token đã bị thu hồi.")
    print("Hãy vào Ngrok Dashboard, copy lại authtoken đầy đủ, chạy lại cell này và dán token mới.")
    raise RuntimeError("Cấu hình Ngrok thất bại vì NGROK_AUTHTOKEN không hợp lệ.") from None

print("Máy chủ mô hình đã được public thành công.")
print("Địa chỉ công khai:", public_url)
print("Trang thông tin:", f"{public_url}/")
print("Swagger:", f"{public_url}/docs")
print("Kiểm tra trạng thái:", f"{public_url}/health")
print("Dự đoán mẫu bằng trình duyệt:", f"{public_url}/predict")
print("Dự đoán bằng Postman:", f"{public_url}/predict")

## 23. Kiểm thử API trực tiếp trong notebook

Gửi request mẫu tới endpoint public của Ngrok. Request và response public dùng các trường tiếng Việt không dấu.

In [ ]:
try:
    with urllib.request.urlopen(f"{public_url}/", timeout=20) as phan_hoi_trang_chu:
        ket_qua_trang_chu = json.loads(phan_hoi_trang_chu.read().decode("utf-8"))
    print("Kết quả mở URL gốc:")
    print(json.dumps(ket_qua_trang_chu, ensure_ascii=False, indent=2))
except urllib.error.URLError as loi:
    print("Không gọi được URL gốc qua Ngrok.")
    print("Vui lòng kiểm tra lại server, Ngrok token và kết nối mạng trong Colab.")
    print(f"Chi tiết lỗi: {loi}")

try:
    with urllib.request.urlopen(f"{public_url}/health", timeout=20) as phan_hoi_suc_khoe:
        ket_qua_suc_khoe = json.loads(phan_hoi_suc_khoe.read().decode("utf-8"))
    print("Kết quả kiểm tra trạng thái máy chủ:")
    print(json.dumps(ket_qua_suc_khoe, ensure_ascii=False, indent=2))
except urllib.error.URLError as loi:
    print("Không gọi được endpoint kiểm tra trạng thái qua Ngrok.")
    print("Vui lòng kiểm tra lại server, Ngrok token và kết nối mạng trong Colab.")
    print(f"Chi tiết lỗi: {loi}")

try:
    with urllib.request.urlopen(f"{public_url}/predict", timeout=20) as phan_hoi_du_doan_mau:
        ket_qua_du_doan_mau = json.loads(phan_hoi_du_doan_mau.read().decode("utf-8"))
    print("Kết quả dự đoán mẫu khi mở /predict bằng trình duyệt:")
    print(json.dumps(ket_qua_du_doan_mau, ensure_ascii=False, indent=2))
except urllib.error.URLError as loi:
    print("Không gọi được endpoint dự đoán mẫu qua Ngrok.")
    print("Vui lòng kiểm tra lại server, Ngrok token và kết nối mạng trong Colab.")
    print(f"Chi tiết lỗi: {loi}")

du_lieu_thu = {
    "dien_tich": 85.5,
    "so_phong": 3,
    "khoang_cach_trung_tam": 5.2,
}

noi_dung_request = json.dumps(du_lieu_thu).encode("utf-8")
yeu_cau = urllib.request.Request(
    f"{public_url}/predict",
    data=noi_dung_request,
    headers={"Content-Type": "application/json"},
    method="POST",
)

try:
    with urllib.request.urlopen(yeu_cau, timeout=20) as phan_hoi:
        ket_qua_api = json.loads(phan_hoi.read().decode("utf-8"))
    print("Kết quả dự đoán:")
    print(f"Diện tích: {du_lieu_thu['dien_tich']} m²")
    print(f"Số phòng: {du_lieu_thu['so_phong']}")
    print(f"Khoảng cách tới trung tâm: {du_lieu_thu['khoang_cach_trung_tam']} km")
    print(f"Giá dự đoán: {ket_qua_api['gia_du_doan']:.2f} triệu VNĐ")
    print(f"Đơn vị API trả về: {ket_qua_api['don_vi']}")
except urllib.error.URLError as loi:
    print("Không gọi được API public qua Ngrok.")
    print("Vui lòng kiểm tra lại server, Ngrok token và kết nối mạng trong Colab.")
    print(f"Chi tiết lỗi: {loi}")

## 24. Hướng dẫn mở trực tiếp bằng trình duyệt và test bằng Postman

Có thể mở trực tiếp các đường dẫn sau trên trình duyệt:

- Trang thông tin máy chủ: `{NGROK_URL}/`
- Kiểm tra trạng thái: `{NGROK_URL}/health`
- Dự đoán mẫu: `{NGROK_URL}/predict`
- Tài liệu Swagger: `{NGROK_URL}/docs`

Dùng thông tin sau để test `POST /predict` từ Postman trên máy khác.

- Phương thức: `POST`
- Địa chỉ: `{NGROK_URL}/predict`
- Header: `Content-Type: application/json`

Nội dung gửi:

```json
{
  "dien_tich": 85.5,
  "so_phong": 3,
  "khoang_cach_trung_tam": 5.2
}
```

Kết quả dự kiến:

```json
{
  "gia_du_doan": 2450.7,
  "don_vi": "trieu_vnd"
}
```

Swagger public nằm tại: `{NGROK_URL}/docs`.

## 25. Hướng dẫn test bằng curl

Thay `{NGROK_URL}` bằng địa chỉ công khai được in ra từ cell cấu hình Ngrok.

Kiểm tra URL gốc:

```bash
curl --location '{NGROK_URL}/'
```

Dự đoán mẫu bằng `GET /predict`:

```bash
curl --location '{NGROK_URL}/predict'
```

Dự đoán bằng `POST /predict`:

```bash
curl --location '{NGROK_URL}/predict' \
--header 'Content-Type: application/json' \
--data '{
  "dien_tich": 85.5,
  "so_phong": 3,
  "khoang_cach_trung_tam": 5.2
}'
```

## 26. Kết luận Bước 3

- Notebook đã bổ sung FastAPI Model Server cho model KNN đã train ở Bước 2.
- API public nhận request bằng các field tiếng Việt không dấu: `dien_tich`, `so_phong`, `khoang_cach_trung_tam`.
- API map request sang feature nội bộ đúng thứ tự: `area`, `rooms`, `distance`.
- Endpoint `GET /` trả thông tin máy chủ và các đường dẫn API.
- Endpoint `GET /health` trả `trang_thai` và `dich_vu`.
- Endpoint `GET /predict` dùng dữ liệu mẫu và trả `du_lieu_dau_vao`, `gia_du_doan`, `don_vi`.
- Endpoint `POST /predict` giữ nguyên contract request/response cho Postman.
- Cả `GET /predict` và `POST /predict` dùng chung logic `loaded_model.predict(...)`, không train lại model.
- Uvicorn chạy trên port 8000 bằng background thread.
- Ngrok public port 8000 và không hard-code auth token.
- Notebook đã có hướng dẫn test bằng Postman và curl.
- Dừng tại Bước 3, chưa làm backend local, Docker hoặc Bước 4.